In [1]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import spacy
import nltk
from nltk.tokenize import word_tokenize

# Sklearn (for later clustering)
from sklearn.preprocessing import LabelEncoder

# Display settingsµ
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Download NLTK data (only runs once)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("All imports loaded successfully")

All imports loaded successfully


In [3]:
# ============================================================
# CELL 2 — Load Core Datasets (Reddit — 70%)
# ============================================================

import os

DATA_PATH = "../../data/processed/"

# --- Load Reddit datasets ---
datasets = {
    "anxiety"      : "reddit_anxiety_cleaned.csv",
    "depression"   : "reddit_depression_cleaned.csv",
    "loneliness"   : "reddit_loneliness_cleaned.csv",
    "mentalhealth" : "reddit_mentalhealth_cleaned.csv",
    "therapy"      : "reddit_therapy_cleaned.csv",
}

dfs = []

for source, filename in datasets.items():
    path = os.path.join(DATA_PATH, filename)
    df = pd.read_csv(path)
    df["source"] = source          # tag where each row came from
    dfs.append(df)
    print(f"✅ {source:15s} → {len(df):>6,} rows | columns: {list(df.columns)}")

# --- Combine into one dataframe ---
reddit_df = pd.concat(dfs, ignore_index=True)

print(f"\n Combined Reddit dataset: {len(reddit_df):,} rows × {reddit_df.shape[1]} columns")
print(f"   Sources: {reddit_df['source'].value_counts().to_dict()}")


✅ anxiety         →    298 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ depression      →    384 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ loneliness      →    139 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ mentalhealth    →    362 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ therapy         →    170 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']

 Combined Reddit dataset: 1,353 rows × 9 columns
   Sources: {'depression': 384, 'mentalhealth': 362, 'anxiety': 298, 'therapy': 170, 'loneliness': 139}


In [ ]:
df.head()

,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,1f788068-ff22-4584-99ee-29c72e9e483b,therapy,reddit_therapy,2026-05-02T23:23:51.791529+00:00,en,"Announcement - Important Update on Bot Advertisements Hello, all! We hope you're having all a lo...",NaN,NaN,NaN
1,8e8f54b5-fefc-4c84-99d3-c5777d0138fd,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"Updated Rules Hello, r/therapy! We hope this post finds you well. We are writing to announce a m...",NaN,NaN,NaN
2,f89c07dc-1f71-4e6b-bb61-1d608edabd29,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"""The ax forgets, but the forest remembers"" Recently I re-read the sentiment, ""The axe forgets bu...",NaN,NaN,NaN
3,822a6d67-2ed7-43ee-9cd6-6b81ebe09cf7,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,Not sure what type of therapist I'm looking for I've finally decided that I need to go through t...,NaN,NaN,NaN
4,64080621-f500-4f30-9909-d7cef2a8f4bf,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"What should I do? I have a brother who is deaf-mute since birth, he is 26 years old now.. My mom...",NaN,NaN,NaN


In [4]:
# ============================================================
# CELL 3 — Load Support Dataset (Counseling — 20%)
# ============================================================

counseling_df = pd.read_csv(os.path.join(DATA_PATH, "hf_mh_counseling_amod_cleaned.csv"))
counseling_df["source"] = "counseling"

print(f"✅ counseling → {len(counseling_df):,} rows | columns: {list(counseling_df.columns)}")
counseling_df.head(2)

✅ counseling → 830 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']


,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,6aa92e1d-7ccc-4afa-bf2c-0e176b16f712,counseling,mh_counseling_amod,2026-05-03T10:40:00.425566+00:00,en,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but t...,NaN,NaN,NaN
1,d0387c40-1d0d-4aa6-87c7-fb768b238e3b,counseling,mh_counseling_amod,2026-05-03T10:40:00.427774+00:00,en,"I have so many issues to address. I have a history of sexual abuse, I'm a breast cancer survivor...",NaN,NaN,NaN


In [5]:
# ============================================================
# CELL 4 — Explore & Preview
# ============================================================

# --- Reddit overview ---
print("=" * 55)
print("📊 REDDIT DATASETS")
print("=" * 55)
print(reddit_df["source"].value_counts())
print(f"\nTotal rows      : {len(reddit_df):,}")
print(f"Null text rows  : {reddit_df['cleaned_text'].isna().sum():,}")
print(f"Avg text length : {reddit_df['cleaned_text'].dropna().str.len().mean():.0f} chars")

# --- Drop null text rows ---
reddit_df = reddit_df.dropna(subset=["cleaned_text"]).reset_index(drop=True)
print(f"\n✅ After dropping nulls: {len(reddit_df):,} rows remain")

# --- Quick sample ---
print("\n--- Sample post (anxiety) ---")
sample = reddit_df[reddit_df["source"] == "anxiety"]["cleaned_text"].iloc[0]
print(sample[:300], "...")

📊 REDDIT DATASETS
source
depression      384
mentalhealth    362
anxiety         298
therapy         170
loneliness      139
Name: count, dtype: int64

Total rows      : 1,353
Null text rows  : 0
Avg text length : 925 chars

✅ After dropping nulls: 1,353 rows remain

--- Sample post (anxiety) ---
Recruiting Moderators! Hey friends, We are looking to grow the team again here on our lovely subreddit. If you are interested, please fill out the form on our application page for r/Anxiety. If you have any questions, feel free to drop them on this post or send us a modmail. Thanks! ...


In [6]:
# ============================================================
# CELL 5 — Clean & Filter Noise
# ============================================================

# --- Noise patterns to drop (mod posts, announcements, etc.) ---
noise_keywords = [
    "recruiting moderators", "mod announcement", "subreddit rules",
    "updated rules", "bot advertisement", "automoderator",
    "weekly thread", "daily thread", "megathread"
]

def is_noise(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in noise_keywords)

# --- Apply filter ---
before = len(reddit_df)
reddit_df = reddit_df[~reddit_df["cleaned_text"].apply(is_noise)].reset_index(drop=True)
after = len(reddit_df)

print(f"  Removed {before - after} noise rows")
print(f" Clean Reddit rows: {after:,}")

# --- Also filter very short posts (less than 50 chars — likely useless) ---
reddit_df = reddit_df[reddit_df["cleaned_text"].str.len() >= 50].reset_index(drop=True)
print(f" After length filter: {len(reddit_df):,} rows remain")

# --- Combine Reddit + Counseling into master df ---
master_df = pd.concat([reddit_df, counseling_df], ignore_index=True)
master_df = master_df[["id", "source", "cleaned_text"]].copy()

print(f"\nMaster dataframe: {len(master_df):,} rows")
print(f"   Sources: {master_df['source'].value_counts().to_dict()}")

  Removed 4 noise rows
 Clean Reddit rows: 1,349
 After length filter: 1,307 rows remain

Master dataframe: 2,137 rows
   Sources: {'counseling': 830, 'depression': 364, 'mentalhealth': 359, 'anxiety': 290, 'therapy': 166, 'loneliness': 128}


In [7]:
# ============================================================
# CELL 6 — NLP-Based Pain Point Classification (Zero-Shot)
# ============================================================

from transformers import pipeline
from collections import Counter

# --- No catch-all labels — every post must map to something specific ---
PAIN_POINT_LABELS = [
    "anxiety and panic attacks",
    "depression and hopelessness",
    "loneliness and social isolation",
    "trauma and abuse",
    "relationship and family issues",
    "low self-worth and self-esteem",
    "suicidal thoughts and self-harm",
    "sleep problems and exhaustion",
    "work stress and burnout",
    "academic pressure and school stress",
    "anger and emotional dysregulation",
    "intrusive thoughts and paranoia",
    "mood disorders and emotional instability",
    "physical symptoms of stress",
    "seeking therapy and professional help",
]

MACRO_MAP = {
    "anxiety and panic attacks":               "anxiety & fear",
    "depression and hopelessness":             "emotional distress",
    "loneliness and social isolation":         "social & relationships",
    "trauma and abuse":                        "trauma & crisis",
    "relationship and family issues":          "social & relationships",
    "low self-worth and self-esteem":          "emotional distress",
    "suicidal thoughts and self-harm":         "trauma & crisis",
    "sleep problems and exhaustion":           "life stressors",
    "work stress and burnout":                 "life stressors",
    "academic pressure and school stress":     "life stressors",
    "anger and emotional dysregulation":       "social & relationships",
    "intrusive thoughts and paranoia":         "anxiety & fear",
    "mood disorders and emotional instability":"emotional distress",
    "physical symptoms of stress":             "anxiety & fear",
    "seeking therapy and professional help":   "help seeking",
}

MACRO_PRIORITY = [
    "trauma & crisis", "emotional distress", "anxiety & fear",
    "social & relationships", "life stressors", "help seeking"
]

print("⏳ Loading zero-shot classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768",
    device=-1
)
print("✅ Classifier ready")

BATCH_SIZE = 32
THRESHOLD = 0.12   # slightly lower so more specific labels fire
TOP_K = 3

pain_points_list = []
texts = master_df["cleaned_text"].fillna("").str[:512].tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    results = classifier(batch, candidate_labels=PAIN_POINT_LABELS, multi_label=True)
    for res in results:
        pairs = list(zip(res["labels"], res["scores"]))
        picked = [label for label, score in pairs if score >= THRESHOLD][:TOP_K]
        # always assign at least the top label — never leave unclassified
        pain_points_list.append(picked if picked else [res["labels"][0]])
    if i % 5 == 0:
        print(f"   {i}/{len(texts)} done...")

master_df["pain_points"]      = pain_points_list
master_df["pain_point_count"] = master_df["pain_points"].apply(len)

# --- Macro groups ---
def assign_macro(pain_points):
    return list(set(MACRO_MAP.get(pp, "help seeking") for pp in pain_points))

def pick_primary_macro(macros):
    for m in MACRO_PRIORITY:
        if m in macros:
            return m
    return macros[0]

master_df["macro_groups"]  = master_df["pain_points"].apply(assign_macro)
master_df["primary_macro"] = master_df["macro_groups"].apply(pick_primary_macro)

# --- Stats ---
print("\n📊 Pain point distribution:")
all_pp = [pp for sublist in master_df["pain_points"] for pp in sublist]
for pain, count in Counter(all_pp).most_common():
    pct = count / len(master_df) * 100
    print(f"   {pain:45s} → {count:>4} ({pct:.1f}%)")

print(f"\n✅ Avg pain points/row : {master_df['pain_point_count'].mean():.2f}")

print("\n📊 Macro group distribution:")
for macro, count in master_df["primary_macro"].value_counts().items():
    pct = count / len(master_df) * 100
    print(f"   {macro:25s} → {count:>4} ({pct:.1f}%) {'█' * int(pct/2)}")

c:\Users\Bichr\miniconda3\envs\data_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading zero-shot classifier...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 754.48it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Classifier ready
   0/2137 done...
   160/2137 done...
   320/2137 done...
   480/2137 done...
   640/2137 done...
   800/2137 done...
   960/2137 done...
   1120/2137 done...
   1280/2137 done...
   1440/2137 done...
   1600/2137 done...
   1760/2137 done...
   1920/2137 done...
   2080/2137 done...

📊 Pain point distribution:
   relationship and family issues                → 1124 (52.6%)
   physical symptoms of stress                   → 1011 (47.3%)
   seeking therapy and professional help         →  589 (27.6%)
   low self-worth and self-esteem                →  492 (23.0%)
   mood disorders and emotional instability      →  484 (22.6%)
   loneliness and social isolation               →  423 (19.8%)
   trauma and abuse                              →  395 (18.5%)
   intrusive thoughts and paranoia               →  282 (13.2%)
   work stress and burnout                       →  282 (13.2%)
   depression and hopelessness                   →  265 (12.4%)
   academic pressure and sch